In [1]:
import pandas as pd
import numpy as np
import seaborn as sns
import matplotlib.pyplot as plt
import optuna
import torch
import matplotlib as mpl
from sklearn.metrics import f1_score
from sklearn.model_selection import train_test_split
from sklearn.neighbors import KNeighborsClassifier
import warnings
warnings.filterwarnings('ignore')


seed = 67

colors = ["#009ED0","#c3c8be","#f2afcb"]


cm = mpl.colors.LinearSegmentedColormap.from_list('colorcitos',colors)



inmutable_df = pd.read_parquet('../data/processed/CLASF_short_data.parquet')

In [2]:
df = inmutable_df.copy()
df = df.reset_index().drop(columns=['index'])
df


,rating,y,w1,w2,w3,w4,w5,w6,w7,w8,...,w759,w760,w761,w762,w763,w764,w765,w766,w767,w768
0,5,gift_cards,-0.024887,0.007899,0.003009,-0.094055,-0.003241,-0.010345,-0.032938,0.025727,...,0.003551,-0.046339,-0.041232,-0.013794,-0.049453,-0.011040,-0.010916,0.034731,0.046359,-0.024646
1,5,digital_music,-0.028520,-0.003986,-0.002436,-0.119818,0.033662,0.002148,-0.031996,0.038514,...,0.050848,-0.014365,0.013234,-0.021985,-0.021832,0.007104,0.013809,0.058802,-0.020029,-0.020273
2,5,gift_cards,0.005387,-0.003584,-0.010338,-0.149245,-0.009772,0.007169,0.015875,-0.000210,...,0.015273,-0.008234,-0.023015,-0.025498,0.005176,-0.010098,0.030468,0.035991,0.011597,-0.038796
3,1,digital_music,-0.020638,-0.052467,-0.014607,-0.099339,0.031590,-0.013430,-0.018903,0.011315,...,0.010960,0.022318,-0.053241,-0.056295,-0.030914,-0.014435,0.016317,0.007946,0.074728,0.033262
4,3,magazines,-0.027466,-0.044827,0.004275,-0.053112,-0.044739,-0.018625,-0.000894,0.030281,...,-0.016350,-0.048549,0.013798,-0.005266,-0.018492,0.027676,0.055425,0.003952,0.028558,0.032652
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
119995,5,gift_cards,-0.048660,-0.014467,-0.016132,-0.116516,0.026375,-0.001256,-0.020911,0.035611,...,0.023353,-0.033935,-0.018241,-0.022677,-0.017755,0.000983,-0.007757,0.044418,0.013258,0.004044
119996,5,gift_cards,-0.009364,0.017477,-0.003649,-0.101146,0.002632,-0.017567,0.051485,0.019261,...,0.016276,0.016702,0.003037,-0.015529,0.037199,0.015329,0.003755,0.027804,0.003915,0.003337
119997,5,gift_cards,-0.011336,-0.012354,0.013837,-0.106397,0.029282,0.019965,-0.000988,0.013854,...,0.041929,-0.052962,-0.015344,-0.037097,-0.007420,0.002436,0.023683,0.004912,-0.013263,-0.020518
119998,5,magazines,-0.040870,-0.049382,0.013034,-0.105640,-0.030041,-0.016332,-0.016006,-0.005308,...,0.004140,-0.014351,-0.024094,-0.051654,-0.029038,0.020465,0.007640,0.073231,-0.001746,0.025817


In [3]:

Y = df['y'].copy()
X = df.drop(columns=['y'])


In [4]:

X_train,X_test,Y_train,Y_test = train_test_split(X,Y,test_size=0.2,random_state=seed,stratify=Y)
X_tunning,X_val,Y_tunnning,Y_val = train_test_split(X_train,Y_train,test_size=0.2,random_state=seed,stratify=Y_train)


# Propocision de modelos

Desde que vimos que hay grupos mas densos en unas regiones que en otros vamos a utilizar KNN


- KNN




In [5]:
def knn_optuna_objective(trial):
    print('lo primero')
    k_n = trial.suggest_int('knn_n_neighbors',2,20,step=2)
    p = trial.suggest_int('p',1,6)
    metric = trial.suggest_categorical('metric',['minkowski','cosine'])

    print('segundo')
    knn = KNeighborsClassifier(n_neighbors=k_n,p=p,metric=metric,n_jobs=1,algorithm='brute')
    print('tercero')
    knn.fit(X_tunning,Y_tunnning)

    y_pred = knn.predict(X_val)

    print('cuarto')

    return f1_score(y_pred=y_pred,y_true=Y_val,average='weighted')


In [6]:
knn_study = optuna.create_study(direction='maximize')
knn_study.optimize(knn_optuna_objective,n_trials=12)

[I 2025-11-20 00:42:16,343] A new study created in memory with name: no-name-a085e6e3-1a29-4246-8863-1e0ebcf16651


lo primero
segundo
tercero


[I 2025-11-20 00:42:41,370] Trial 0 finished with value: 0.8947071519876674 and parameters: {'knn_n_neighbors': 20, 'p': 1, 'metric': 'cosine'}. Best is trial 0 with value: 0.8947071519876674.


cuarto
lo primero
segundo
tercero


[I 2025-11-20 00:43:04,392] Trial 1 finished with value: 0.896461065167397 and parameters: {'knn_n_neighbors': 12, 'p': 5, 'metric': 'cosine'}. Best is trial 1 with value: 0.896461065167397.


...

cuarto
lo primero
segundo
tercero


[I 2025-11-20 00:46:10,044] Trial 10 finished with value: 0.874738879779785 and parameters: {'knn_n_neighbors': 2, 'p': 2, 'metric': 'minkowski'}. Best is trial 9 with value: 0.9005120648051543.


cuarto
lo primero
segundo
tercero


[I 2025-11-20 01:26:39,731] Trial 11 finished with value: 0.8972742139964527 and parameters: {'knn_n_neighbors': 14, 'p': 6, 'metric': 'minkowski'}. Best is trial 9 with value: 0.9005120648051543.


cuarto


In [7]:
knn_study.best_params

{'knn_n_neighbors': 16, 'p': 2, 'metric': 'minkowski'}

In [13]:
knn_study.best_trial

FrozenTrial(number=9, state=1, values=[0.9005120648051543], datetime_start=datetime.datetime(2025, 11, 20, 0, 45, 48, 378756), datetime_complete=datetime.datetime(2025, 11, 20, 0, 45, 59, 175669), params={'knn_n_neighbors': 16, 'p': 2, 'metric': 'minkowski'}, user_attrs={}, system_attrs={}, intermediate_values={}, distributions={'knn_n_neighbors': IntDistribution(high=20, log=False, low=2, step=2), 'p': IntDistribution(high=6, log=False, low=1, step=1), 'metric': CategoricalDistribution(choices=('minkowski', 'cosine'))}, trial_id=9, value=None)

In [8]:

knn = KNeighborsClassifier(n_neighbors=5,p=2,metric='cosine')



In [9]:

knn.fit(X_tunning,Y_tunnning)


KNeighborsClassifier(metric='cosine')

In [10]:

y_pred = knn.predict(X_val)

f1_score(y_pred=y_pred,y_true=Y_val,average='weighted')


0.8935340672020057

### KNN

#### Logistic regresion

In [11]:
def log_reg_optuna_objective(trial):
    c = trial.suggest_float('log_reg_C',0,3)
    p = trial.suggest_int('p',1,6)
    metric = trial.suggest_categorical('metric',['minkowski','cosine'])

    knn = KNeighborsClassifier(n_neighbors=k_n,p=p,metric=metric,n_jobs=-1,algorithm='brute')
    knn.fit(X_tunning,Y_tunnning)

    y_pred = knn.predict(X_val)

    return f1_score(y_pred=y_pred,y_true=Y_val,average='weighted')


In [12]:

def knn_optuna_objective(trial):
    k_n = trial.suggest_int('knn_n_neighbors',2,20,step=2)
    c = trial.suggest_int('p',1,6)
    metric = trial.suggest_categorical('metric',['minkowski','cosine'])

    knn = KNeighborsClassifier(n_neighbors=k_n,p=p,metric=metric,n_jobs=-1)
    knn.fit(X_tunning,Y_tunnning)

    y_pred = knn.predict(X_val)

    return f1_score(y_pred=y_pred,y_true=Y_val,average='weighted')



# Maybe PCA